# 01 — GME day-ahead prices (MGP)

Reads the GME yearly files and builds the **hourly price series**: the national price (PUN) and the 7 bidding zones.

- **Input:** `data/raw/gme_mgp/Anno2021.zip` … `Anno2025.zip`
- **Output:** `data/processed/mgp_prices_hourly_2021_2025q3.parquet` (41,615 hours, 26 columns)
- **Key points:** GME numbers hours 1–25 to handle daylight saving time, so timestamps are built in UTC; the series is cut at 30 Sep 2025 because the market moved to 15-minute products on 1 Oct 2025.

In [1]:
# read the gme_mgp data from the raw folder and print some info about it
from pathlib import Path
import zipfile
import pandas as pd

RAW = Path.cwd().parent / "data" / "raw" / "gme_mgp"

def read_prices(zip_path: Path) -> pd.DataFrame:
    """Estrae il foglio Prezzi-Prices dall'xlsx orario contenuto nello ZIP."""
    with zipfile.ZipFile(zip_path) as z:
        xlsx = [n for n in z.namelist()
                if n.endswith(".xlsx") and not n.endswith("_15.xlsx")]
        assert len(xlsx) == 1, f"attesi 1 xlsx orario, trovati: {xlsx}"
        with z.open(xlsx[0]) as f:
            return pd.read_excel(f, sheet_name="Prezzi-Prices")

df21 = read_prices(RAW / "Anno2021.zip")

print(df21.shape)
print(repr(list(df21.columns)))
print(df21.dtypes.head(4))
print(df21.head(3).iloc[:, :5])

(8760, 23)
['   Data/Date\n(YYYYMMDD)', 'Ora\n/Hour', 'PUN', 'AUST', 'BSP', 'CALA', 'CNOR', 'COAC', 'CORS', 'CSUD', 'FRAN', 'GREC', 'MALT', 'MONT', 'NORD', 'SARD', 'SICI', 'SLOV', 'SUD', 'SVIZ', 'XAUS', 'XFRA', 'XGRE']
   Data/Date\n(YYYYMMDD)      int64
Ora\n/Hour                    int64
PUN                         float64
AUST                        float64
dtype: object
      Data/Date\n(YYYYMMDD)  Ora\n/Hour       PUN      AUST       BSP
0                  20210101           1  50.87000  50.87000  50.87000
1                  20210101           2  48.19000  48.19000  48.19000
2                  20210101           3  44.67966  44.67966  44.67966


In [2]:
# normalize the data
import re

def normalize(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c).strip() for c in df.columns]
    df.columns = ["date", "hour", "national_price"] + list(df.columns[3:])
    df["data"] = pd.to_numeric(df["date"], errors="raise").astype("int64")
    df["ora"]  = pd.to_numeric(df["hour"],  errors="raise").astype("int64")
    return df

df21 = normalize(df21)
print(df21.columns[:5].tolist())

ore = df21.groupby("date")["hour"].agg(ore="count", max_ora="max")
print(ore["ore"].value_counts())
print(ore[ore["ore"] != 24])

['date', 'hour', 'national_price', 'AUST', 'BSP']
ore
24    363
23      1
25      1
Name: count, dtype: int64
          ore  max_ora
date                  
20210328   23       23
20211031   25       25


In [3]:
# --- 1. local midnight of each day ---------------------------------------
# 'data' is an int yyyymmdd -> string -> naive datetime (midnight).
# tz_localize anchors it to Italian time: local midnight is NEVER
# ambiguous nor non-existent (DST switches happen at 02:00/03:00).
day_local = (
    pd.to_datetime(df21["data"].astype(str), format="%Y%m%d")
      .dt.tz_localize("Europe/Rome")
)

# --- 2. move to UTC BEFORE adding the hours ------------------------------
# In UTC every hour is always 60 minutes long: the arithmetic is safe.
# In local time, on 31 Oct 02:00 exists twice -> adding timedeltas to
# local timestamps would silently produce duplicates or gaps.
start_utc = day_local.dt.tz_convert("UTC")

# hour=1 is the first period of the day -> offset 0. Hence the -1.
df21["ts_utc"] = start_utc + pd.to_timedelta(df21["ora"] - 1, unit="h")

# Local time: for readability and plots only, never as a key.
df21["ts_local"] = df21["ts_utc"].dt.tz_convert("Europe/Rome")

# --- 3. sanity checks: if any of these fails, the timestamp is wrong ------
print("rows:", len(df21))
print("unique timestamps:", df21["ts_utc"].nunique())          # must be 8760
print("monotonic increasing:", df21["ts_utc"].is_monotonic_increasing)
print("gaps other than 1h:", (df21["ts_utc"].diff().dropna()
                              != pd.Timedelta("1h")).sum())     # must be 0

# --- 4. eyeball check on the two DST days --------------------------------
for g in (20210328, 20211031):
    print(f"\n--- {g} ---")
    print(df21.loc[df21["data"] == g, ["ora", "ts_utc", "ts_local"]]
              .head(6).to_string(index=False))

rows: 8760
unique timestamps: 8760
monotonic increasing: True
gaps other than 1h: 0

--- 20210328 ---
 ora                    ts_utc                  ts_local
   1 2021-03-27 23:00:00+00:00 2021-03-28 00:00:00+01:00
   2 2021-03-28 00:00:00+00:00 2021-03-28 01:00:00+01:00
   3 2021-03-28 01:00:00+00:00 2021-03-28 03:00:00+02:00
   4 2021-03-28 02:00:00+00:00 2021-03-28 04:00:00+02:00
   5 2021-03-28 03:00:00+00:00 2021-03-28 05:00:00+02:00
   6 2021-03-28 04:00:00+00:00 2021-03-28 06:00:00+02:00

--- 20211031 ---
 ora                    ts_utc                  ts_local
   1 2021-10-30 22:00:00+00:00 2021-10-31 00:00:00+02:00
   2 2021-10-30 23:00:00+00:00 2021-10-31 01:00:00+02:00
   3 2021-10-31 00:00:00+00:00 2021-10-31 02:00:00+02:00
   4 2021-10-31 01:00:00+00:00 2021-10-31 02:00:00+01:00
   5 2021-10-31 02:00:00+00:00 2021-10-31 03:00:00+01:00
   6 2021-10-31 03:00:00+00:00 2021-10-31 04:00:00+01:00


In [4]:
def add_timestamp(df: pd.DataFrame) -> pd.DataFrame:
    """Build a UTC timestamp from the (date, hour) pair. DST-safe."""
    # Local midnight is never ambiguous -> safe to localize, then move to
    # UTC and do the hour arithmetic there, where every hour is 60 minutes.
    start_utc = (
        pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
          .dt.tz_localize("Europe/Rome")
          .dt.tz_convert("UTC")
    )
    df = df.copy()
    # hour=1 is the first period of the day -> offset 0. Hence the -1.
    df["ts_utc"] = start_utc + pd.to_timedelta(df["hour"] - 1, unit="h")
    # Local time: for readability and plots only, never as a key.
    df["ts_local"] = df["ts_utc"].dt.tz_convert("Europe/Rome")
    return df


def load_year(zip_path) -> pd.DataFrame:
    """Full per-year pipeline: read -> normalise -> timestamp -> provenance."""
    raw = read_prices(zip_path)

    # Keep the ORIGINAL name of column 2 before renaming it: it tells us
    # whether the row carries PUN (2021-24) or PUN INDEX GME (2025).
    # This is data lineage, not decoration - the two are different series.
    source = re.sub(r"\s+", " ", str(raw.columns[2])).strip()

    df = add_timestamp(normalize(raw))
    df["price_source"] = pd.Categorical([source] * len(df))
    return df


# Load all five years, keeping them separate for now.
years = {y: load_year(RAW / f"Anno{y}.zip") for y in range(2021, 2026)}

# Per-year diagnostics: any anomaly must surface HERE, not after concat.
for y, df in years.items():
    gaps = (df["ts_utc"].diff().dropna() != pd.Timedelta("1h")).sum()
    print(f"{y} | rows={len(df):5d} | cols={df.shape[1]} | "
          f"source={df['price_source'].iloc[0]:14s} | "
          f"monotonic={df['ts_utc'].is_monotonic_increasing} | "
          f"gaps={gaps} | NaN price={df['national_price'].isna().sum()}")

# Do all years share the same set of columns? A bidding zone missing in one
# year would silently turn into an all-NaN column after concatenation.
reference_cols = set(years[2021].columns)
for y, df in years.items():
    diff = reference_cols.symmetric_difference(df.columns)
    print(y, "column diff vs 2021:", diff if diff else "none")

2021 | rows= 8760 | cols=28 | source=PUN            | monotonic=True | gaps=0 | NaN price=0
2022 | rows= 8760 | cols=28 | source=PUN            | monotonic=True | gaps=0 | NaN price=0
2023 | rows= 8760 | cols=28 | source=PUN            | monotonic=True | gaps=0 | NaN price=0
2024 | rows= 8784 | cols=28 | source=PUN            | monotonic=True | gaps=0 | NaN price=0
2025 | rows= 8760 | cols=28 | source=PUN INDEX GME  | monotonic=True | gaps=0 | NaN price=0
2021 column diff vs 2021: none
2022 column diff vs 2021: none
2023 column diff vs 2021: none
2024 column diff vs 2021: none
2025 column diff vs 2021: none


In [5]:
# Which columns do we actually carry? Expected 26:
# 23 from the source sheet + ts_utc + ts_local + price_source.
print(len(years[2021].columns))
print(list(years[2021].columns))

28
['date', 'hour', 'national_price', 'AUST', 'BSP', 'CALA', 'CNOR', 'COAC', 'CORS', 'CSUD', 'FRAN', 'GREC', 'MALT', 'MONT', 'NORD', 'SARD', 'SICI', 'SLOV', 'SUD', 'SVIZ', 'XAUS', 'XFRA', 'XGRE', 'data', 'ora', 'ts_utc', 'ts_local', 'price_source']


In [6]:
def normalize(df: pd.DataFrame) -> pd.DataFrame:
    """Clean up column names and enforce integer types on date/hour."""
    df = df.copy()

    # Header cells contain newlines -> collapse any whitespace run.
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]

    # Rename the first three columns BY POSITION: column 2 is named 'PUN'
    # up to 2024 and 'PUN INDEX GME' in 2025, so a name-based rename would
    # silently do nothing on the 2025 file.
    df.columns = ["date", "hour", "national_price"] + list(df.columns[3:])

    # Coerce in place. errors="raise" so that a year with text in these
    # columns fails loudly instead of producing NaN that flows downstream.
    df["date"] = pd.to_numeric(df["date"], errors="raise").astype("int64")
    df["hour"] = pd.to_numeric(df["hour"], errors="raise").astype("int64")
    return df


# Reload everything with the fixed function.
years = {y: load_year(RAW / f"Anno{y}.zip") for y in range(2021, 2026)}

# 23 source columns + ts_utc + ts_local + price_source = 26. Fail loudly.
for y, df in years.items():
    assert df.shape[1] == 26, f"{y}: {df.shape[1]} columns, expected 26"
print("all years:", years[2021].shape[1], "columns")
print(list(years[2021].columns))

all years: 26 columns
['date', 'hour', 'national_price', 'AUST', 'BSP', 'CALA', 'CNOR', 'COAC', 'CORS', 'CSUD', 'FRAN', 'GREC', 'MALT', 'MONT', 'NORD', 'SARD', 'SICI', 'SLOV', 'SUD', 'SVIZ', 'XAUS', 'XFRA', 'XGRE', 'ts_utc', 'ts_local', 'price_source']


In [7]:
# One frame, chronologically ordered. ignore_index because the per-year
# indices all restart at 0 and would collide.
prices = (
    pd.concat(years.values(), ignore_index=True)
      .sort_values("ts_utc")
      .reset_index(drop=True)
)

# Perimeter end = first instant of the quarter-hourly MTU regime, i.e.
# 1 Oct 2025 00:00 ITALIAN time. In UTC that is 30 Sep 2025 22:00, so a
# naive cut at "ts_utc < 2025-10-01" would drag in two hours of the new
# market design. Always express the cut in the market's own timezone.
CUTOFF_UTC = pd.Timestamp("2025-10-01 00:00", tz="Europe/Rome").tz_convert("UTC")
prices = prices.loc[prices["ts_utc"] < CUTOFF_UTC].reset_index(drop=True)

# --- invariants of the final series -------------------------------------
assert prices["ts_utc"].is_unique,               "duplicate timestamps"
assert prices["ts_utc"].is_monotonic_increasing, "not sorted"
assert (prices["ts_utc"].diff().dropna() == pd.Timedelta("1h")).all(), "gaps"

# Expected length, computed rather than hard-coded: elapsed UTC hours
# between the first timestamp and the cutoff.
expected = int((CUTOFF_UTC - prices["ts_utc"].iloc[0]) / pd.Timedelta("1h"))
print("rows:", len(prices), "| expected:", expected)

print("from:", prices["ts_local"].iloc[0], " to:", prices["ts_local"].iloc[-1])
print(prices["price_source"].value_counts())
print(prices["national_price"].describe())

rows: 41615 | expected: 41615
from: 2021-01-01 00:00:00+01:00  to: 2025-09-30 23:00:00+02:00
price_source
PUN              35064
PUN INDEX GME     6551
Name: count, dtype: int64
count    41615.000000
mean       158.410059
std        106.883417
min          0.000000
25%         98.068210
50%        122.150000
75%        184.911140
max        870.000000
Name: national_price, dtype: float64


In [8]:
zeros = prices.loc[prices["national_price"] == 0]
print("zero-price hours:", len(zeros), f"({len(zeros)/len(prices):.3%})")

# Are there negative prices at all? Matters: it tells us whether the series
# is censored at zero or genuinely free to go below.
print("negative-price hours:", (prices["national_price"] < 0).sum())

# When do they happen? Year, month and hour-of-day are the three cuts that
# separate a market phenomenon from a data glitch.
print("\nby year:\n",  zeros.groupby(zeros["ts_local"].dt.year).size())
print("\nby month:\n", zeros.groupby(zeros["ts_local"].dt.month).size())
print("\nby hour:\n",  zeros.groupby(zeros["ts_local"].dt.hour).size())

# Zonal prices during those hours: if the national price is zero because
# every zone is zero, it is a system-wide oversupply event.
print("\n", zeros[["ts_local", "national_price",
                   "NORD", "CNOR", "CSUD", "SUD", "SICI", "SARD"]]
        .head(8).to_string(index=False))

zero-price hours: 10 (0.024%)
negative-price hours: 0

by year:
 ts_local
2025    10
dtype: int64

by month:
 ts_local
5    10
dtype: int64

by hour:
 ts_local
11    1
12    1
13    2
14    3
15    2
16    1
dtype: int64

                  ts_local  national_price  NORD  CNOR  CSUD  SUD  SICI  SARD
2025-05-01 14:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-18 13:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-18 14:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-18 15:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-25 11:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-25 12:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-25 13:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0
2025-05-25 14:00:00+02:00             0.0   0.0   0.0   0.0  0.0   0.0   0.0


In [9]:
PROCESSED = Path.cwd().parent / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Parquet, not CSV: it keeps the timezone-aware dtype of ts_utc/ts_local and
# the categorical price_source as they are. A CSV round-trip would turn them
# back into strings and you would have to re-parse (and re-localize) them
# every single time - which is exactly where DST bugs creep back in.
out = PROCESSED / "mgp_prices_hourly_2021_2025q3.parquet"
prices.to_parquet(out, index=False)

# Read it back and prove the dtypes survived. Never trust a write you
# haven't read back.
check = pd.read_parquet(out)
print(out.name, f"{out.stat().st_size/1e6:.1f} MB")
print(check.dtypes[["ts_utc", "ts_local", "price_source", "national_price"]])
print(check["ts_utc"].equals(prices["ts_utc"]))

mgp_prices_hourly_2021_2025q3.parquet 4.7 MB
ts_utc                    datetime64[us, UTC]
ts_local          datetime64[us, Europe/Rome]
price_source                              str
national_price                        float64
dtype: object
True
